# Module 3 · Generative AI & LLMs — Live GPU Demo
**Prof. Dr. R. Bhaduri · DSU_NVIDIA AI-First Core Team**

Four demos, each mapped to the Module 3 slides. Run them top to bottom.

| Cell | Module 3 concept | Slides |
|------|------------------|--------|
| 1 | Word embeddings — distance encodes meaning | 12–13 |
| 2 | Next-word prediction (context → the pangram) | 15–21 |
| 3 | Self-attention inside the Transformer | 18–19 |
| 4 | One foundation model, many tasks | 7, 9, 22 |


## 0 · Verify the GPU
Confirms we are really running on an NVIDIA GPU (the 'GPU in the cloud' Brev gives us).

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", device)

## 1 · Words as Vectors — *distance encodes meaning*  (Slides 12–13)
Most ML can't read raw text — it needs numbers. A **word embedding** turns each word into a
vector so that *geometry* captures meaning. We reproduce two slide claims live:
related words sit close, and country→capital pairs share the same vector **direction**.

In [ ]:
import gensim.downloader as api
print("Loading GloVe embeddings (cached on the instance)...")
wv = api.load("glove-wiki-gigaword-100")   # 100-d vectors

# (a) Related words cluster together
for a,b in [("compute","calculate"), ("microwave","mixer"), ("king","cat")]:
    print(f"similarity({a:10s},{b:10s}) = {wv.similarity(a,b):.3f}")

print()
# (b) Analogy = geometry:  king - man + woman = ?   ;  France - Paris + Berlin = ?
print("king  - man    + woman  ->", wv.most_similar(positive=['king','woman'],  negative=['man'])[0])
print("paris - france + germany ->", wv.most_similar(positive=['paris','germany'], negative=['france'])[0])
print("london - england + japan ->", wv.most_similar(positive=['london','japan'], negative=['england'])[0])
print("\nParallel arrows on the slide = the same 'is-capital-of' direction in vector space.")

## 2 · Next-Word Prediction — *context changes the guess*  (Slides 15–21)
The slide sequence: with **no context** the model guesses blindly; with **full context** the
Transformer completes the classic pangram. We show both with GPT-2.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()

def top_next_words(prompt, k=5):
    ids = tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = gpt2(**ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(tok.decode(i).strip(), float(p)) for i, p in zip(top.indices, top.values)]

print("Weak context  'lazy'                         ->", top_next_words("lazy"))
print("Full context  'The quick brown fox ... lazy' ->",
      top_next_words("The quick brown fox jumps over the lazy"))

## 3 · Self-Attention — *the Transformer's trick*  (Slides 18–19)
'Attention Is All You Need' (Vaswani et al., 2017). The heat-map shows, for each word,
**which earlier words it attends to** — the Multi-Head Attention block, made visible.

In [ ]:
from bertviz import head_view
from transformers import AutoTokenizer, AutoModel

m_name = "gpt2"
atok = AutoTokenizer.from_pretrained(m_name)
amodel = AutoModel.from_pretrained(m_name, output_attentions=True).eval()

sentence = "The quick brown fox jumps over the lazy dog"
inputs = atok(sentence, return_tensors="pt")
attn = amodel(**inputs).attentions
tokens = atok.convert_ids_to_tokens(inputs["input_ids"][0])
head_view(attn, tokens)   # interactive: pick a layer, hover a word

## 4 · One Foundation Model, Many Tasks  (Slides 7, 9, 22)
A foundation model / LLM is flexible: **one** model summarises, translates and answers.
We use FLAN-T5 locally (no API key needed). *Optional:* swap in an NVIDIA NIM from
build.nvidia.com by setting `NVIDIA_API_KEY` — see the commented block.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

t5tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(device).eval()

def ask(prompt, max_new_tokens=60):
    ids = t5tok(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = t5.generate(**ids, max_new_tokens=max_new_tokens)
    return t5tok.decode(out[0], skip_special_tokens=True)

print("SUMMARISE :", ask("Summarize: A foundation model is trained once on broad, "
                         "largely unlabelled data, then adapted to many downstream tasks."))
print("TRANSLATE :", ask("Translate to German: Generative AI creates new content from a prompt."))
print("Q & A     :", ask("Answer: What architecture introduced self-attention in 2017?"))

In [ ]:
# --- OPTIONAL: run the same three tasks against an NVIDIA NIM (build.nvidia.com) ---
# import os
# from openai import OpenAI
# client = OpenAI(base_url="https://integrate.api.nvidia.com/v1",
#                 api_key=os.environ["NVIDIA_API_KEY"])
# r = client.chat.completions.create(
#     model="meta/llama-3.1-8b-instruct",
#     messages=[{"role":"user","content":"Summarize what a foundation model is in one line."}])
# print(r.choices[0].message.content)

## Recap
You have now *seen* every box from the module: words → vectors, context → prediction,
attention → the Transformer, and one foundation model doing many tasks.
Stop this instance when finished to save GPU credits.